In [1]:
# from ultralytics import SAM
from ultralytics.models.sam import Predictor as SAMPredictor

# Load a model
# model = SAM("sam_b.pt")
overrides = {"conf": 0.25, "task": "segment", "mode": "predict", "imgsz": 640, "model": "mobile_sam.pt"}
predictor = SAMPredictor(overrides=overrides)

In [2]:
# %pip install segment-anything

In [3]:
# import urllib


# urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth", "sam_vit_h_4b8939.pth")

In [4]:
import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)

In [15]:
from glob import glob
import os

import json

import cv2
from matplotlib import pyplot as plt
from tqdm import tqdm

import numpy as np
from PIL import Image, ImageDraw

import geopandas as gpd

In [6]:
def calculateIoU(gtMask, predMask):
        # Calculate the true positives,
        # false positives, and false negatives
        tp = 0
        fp = 0
        fn = 0

        for i in range(len(gtMask)):
            for j in range(len(gtMask[0])):
                if gtMask[i][j] == 1 and predMask[i][j] == 1:
                    tp += 1
                elif gtMask[i][j] == 0 and predMask[i][j] == 1:
                    fp += 1
                elif gtMask[i][j] == 1 and predMask[i][j] == 0:
                    fn += 1

        # Calculate IoU
        iou = tp / (tp + fp + fn)

        return iou

In [7]:
def show_mask(mask,ax, random_color=False):

    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    #return mask_image

def show_mask_box(mask, ax, random_color=False):

    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))



In [8]:
width = 640
height = 640

def convert_polygon_to_mask(geo):
      gtmask=np.zeros((width,height))
      for orig_row in geo:
            polygon=[]

            if orig_row.geom_type=="Polygon":
                for point in orig_row.exterior.coords:
                    polygon.append(point)
                img = Image.new('L', (width, height), 0)
                ImageDraw.Draw(img).polygon(polygon, outline=1, fill=1)
                gt_mask_building = np.array(img)
                gtmask=gtmask+gt_mask_building
            else:
                for x in orig_row.geoms:
                 for point in x.exterior.coords:
                    polygon.append(point)

                img = Image.new('L', (width, height), 0)
                ImageDraw.Draw(img).polygon(polygon, outline=1, fill=1)
                gt_mask_building = np.array(img)
                gtmask=gtmask+gt_mask_building
      return gtmask


In [ ]:
images = r'C:\Users\alias\Documents\UNI\Thesis\GreenHouse-Dataset_Images+jsonlabels+masks\Greenhousedataset4\images'
masks = r'C:\Users\alias\Documents\UNI\Thesis\GreenHouse-Dataset_Images+jsonlabels+masks\Greenhousedataset4\masks'

flag=0

for image, shape, mask in tqdm(zip(glob(images+'/*.png')[:1], glob(images+'/*.json'),glob(masks+'/*.png'))):

    name=image.split('.')[0]

    image = cv2.imread(image)
    mask = cv2.imread(mask)
    f = json.load(open(shape))
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    points = [shape['points'] for shape in f['shapes']]
    lab=np.array([1])

    predictor.set_image(image)
    
    mask_tile=np.zeros((3,512,512))
    mask_tile0=np.zeros((512,512))
    mask_tile1=np.zeros((512,512))
    mask_tile2=np.zeros((512,512))
    for i in points:
        i = np.array(i, dtype=np.float32).reshape(-1, 2)   # shape (N, 2)
        lab = np.ones(len(i), dtype=np.int64)              # shape (N,), 1 = foreground
        masks, scores, logits = predictor.predict(
            point_coords=i,
            point_labels=lab,
            multimask_output=True,
        )
        masks=np.array(masks)
        mask_tile=mask_tile+masks


        msk0=masks[0]
        mask_tile0=mask_tile0+msk0
        msk1=masks[1]
        mask_tile1=mask_tile1+msk1
        msk2=masks[2]
        mask_tile2=mask_tile2+msk2

        print("masks:", len(masks))

    iou0=calculateIoU(mask,mask_tile0)
    iou1=calculateIoU(mask,mask_tile1)
    iou2=calculateIoU(mask,mask_tile2)

    for i,(mask,score) in enumerate(zip(mask_tile,scores)):
        plt.figure(figsize=(10,10))
        plt.imshow(image)
        show_mask(mask, plt.gca())
        # show_points(input_point, input_label, plt.gca())
        plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()


    for i,orig_g in zip(glob.glob(shapefile+'/'+name),glob.glob(orig_shp+'/'+name)):
        input_point=[]
        input_label=[]
        mask_tile=np.zeros((3,512,512))
        mask_tile0=np.zeros((512,512))
        mask_tile1=np.zeros((512,512))
        mask_tile2=np.zeros((512,512))
        score_tile_buil=[]
        score_tile=[]
        gt_m=np.zeros((512,512))
        try:
          f=gpd.read_file(i)
          geo=f['geometry']
          f1=gpd.read_file(orig_g)
          geo1=f1['geometry']
          object_id=f['OBJECTID']
          gt_m=convert_polygon_to_mask(geo1)

        except Exception as e :
          print(e)
          flag=1
          break
        count=0

        for row,oid in zip(geo,object_id):

            x=row.x
            y=row.y
            i=[x,y]


            input_point.append(i)
            input_label.append(1)
            i=np.array([i])
            lab=np.array([1])

            masks, scores, logits = predictor.predict(
            point_coords=i,
            point_labels=lab,
            multimask_output=True,
            )
            masks=np.array(masks)
            mask_tile=mask_tile+masks


            msk0=masks[0]
            mask_tile0=mask_tile0+msk0
            msk1=masks[1]
            mask_tile1=mask_tile1+msk1
            msk2=masks[2]
            mask_tile2=mask_tile2+msk2

        iou0=calculateIoU(gt_m,mask_tile0)
        iou1=calculateIoU(gt_m,mask_tile1)
        iou2=calculateIoU(gt_m,mask_tile2)


        scores=[iou0,iou1,iou2]
        if flag==1:
          flag=0
          continue
        input_point=np.array(input_point)
        input_label=np.array(input_label)

        for i,(mask,score) in enumerate(zip(mask_tile,scores)):
            plt.figure(figsize=(10,10))
            plt.imshow(image)
            show_mask(mask, plt.gca())
            show_points(input_point, input_label, plt.gca())
            plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
            plt.axis('off')
            plt.show()



0it [00:01, ?it/s]


ValueError: operands could not be broadcast together with shapes (3,512,512) (3,408,632) 